# 🔬 ONN-DETECT: EmbeddingGemma 2 vs. SigLIP-B/16 Benchmark (Colab T4)
### Side-by-Side Frozen Forensic Probe Evaluation Under Strict 4-Split Protocol

This notebook benchmarks **EmbeddingGemma 2 (440M Text+Vision)** against the shipped **SigLIP-B/16 Foundation Probe** using ONN-DETECT's strict group-disjoint evaluation protocol:
- **`stacking` split**: Fit binary regularized linear probe $p_{\text{AI}} = \sigma(w^T e + b)$ ONLY.
- **`fusion_val` split**: Select regularization strength $C$ ONLY.
- **`calibration` split**: Select decision threshold at $\text{FPR} \le 0.05$ ONLY.
- **`test` split**: Read once at the end to report sealed ROC-AUC, EER, TPR@1%FPR, TPR@0.1%FPR, and per-source real FPR.

> **FP32 Precision Requirement on T4 GPUs**: NVIDIA T4 (Turing, Compute Capability 7.5) lacks native `bfloat16` hardware units. Running EmbeddingGemma 2 in standard `float16` causes numerical underflow/NaN and representation collapse. This benchmark strictly enforces `torch.float32` and passes `audio_config=None` to avoid loading unneeded audio weights.

--- 
## Step 1: Verify NVIDIA T4 GPU Environment

In [ ]:
!nvidia-smi

import torch
print("=" * 60)
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active Device   : {torch.cuda.get_device_name(0)}")
    print(f"VRAM Total      : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
print("=" * 60)

--- 
## Step 2: Install Dependencies & Setup Environment

In [ ]:
!pip install -U sentence-transformers transformers scikit-learn pillow numpy matplotlib

# Clone or link the Onn-Detect- repository if not already present in workspace
import os
from pathlib import Path

if not Path("ml/scripts/evaluate_embeddinggemma2_probe.py").exists():
    print("Cloning Onn-Detect- repository...")
    !git clone https://github.com/shubhamiitshk/Onn-Detect-.git /content/Onn-Detect-
    os.chdir("/content/Onn-Detect-")

print(f"Current working directory: {os.getcwd()}")

--- 
## Step 3: Configure Dataset Location
Point to your group-disjoint `openfake_visual` corpus containing `stacking`, `fusion_val`, `calibration`, and `test` splits (or generate synthetic splits for quick verification).

In [ ]:
from pathlib import Path

DATA_ROOT = Path("/content/datasets/openfake_visual")
OUTPUT_DIR = Path("/content/runs/probe_benchmark")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# If dataset directory does not exist, prepare a fast demonstration pilot split
if not (DATA_ROOT / "stacking").exists():
    print(f"{DATA_ROOT} not found. Preparing pilot forensic corpus via prepare_forensic_pilot_corpus.py...")
    !python ml/scripts/prepare_forensic_pilot_corpus.py --output-dir {DATA_ROOT}
else:
    print(f"Found dataset splits at {DATA_ROOT}")

--- 
## Step 4: Run Baseline SigLIP-B/16 Probe Evaluation

In [ ]:
SIGLIP_OUT = OUTPUT_DIR / "siglip_baseline"
SIGLIP_OUT.mkdir(parents=True, exist_ok=True)

!python -m ml.scripts.evaluate_vfm_probes \
    --data-root {DATA_ROOT} \
    --output-dir {SIGLIP_OUT} \
    --towers google/siglip-base-patch16-224

--- 
## Step 5: Run EmbeddingGemma 2 Benchmark in FP32

In [ ]:
GEMMA_OUT = OUTPUT_DIR / "embeddinggemma2_eval"
GEMMA_OUT.mkdir(parents=True, exist_ok=True)

!python ml/scripts/evaluate_embeddinggemma2_probe.py \
    --data-root {DATA_ROOT} \
    --output-dir {GEMMA_OUT} \
    --model-name unsloth/embeddinggemma-2 \
    --device cuda

--- 
## Step 6: Side-by-Side Comparison & Decision Gate

In [ ]:
import json

gemma_report_path = GEMMA_OUT / "embeddinggemma2_evaluation_report.json"
if gemma_report_path.exists():
    gemma_report = json.loads(gemma_report_path.read_text(encoding="utf-8"))
    gemma_test = gemma_report["splits"]["test"]
    
    print("=" * 75)
    print("                 FROZEN VISION PROBE BENCHMARK SUMMARY                 ")
    print("=" * 75)
    print(f"{'Metric':<25} | {'SigLIP-B/16 (Shipped)':<22} | {'EmbeddingGemma 2':<20}")
    print("-" * 75)
    print(f"{'Backbone Parameters':<25} | {'86M (Vision Only)':<22} | {'440M (170M V + 270M T)':<20}")
    print(f"{'Asset Size (Quantized)':<25} | {'89.96 MB .tflite':<22} | {'309.6 - 388 MB .litertlm':<20}")
    print(f"{'Linear Head Test AUC':<25} | {'0.8919':<22} | {gemma_test['auc']:<20.4f}")
    print(f"{'TAP Head Test AUC':<25} | {'0.9921':<22} | {'N/A (Global Only)':<20}")
    print(f"{'Test TPR @ 1% FPR':<25} | {'0.7243 (TAP)':<22} | {gemma_test['tpr_at_fpr_1pct']:<20.4f}")
    print(f"{'Test EER':<25} | {'0.0750 (TAP)':<22} | {gemma_test['eer']:<20.4f}")
    print("=" * 75)
    
    # Automated Decision Gate
    print("\n--- DECISION GATE ANALYSIS ---")
    if gemma_test['auc'] > 0.9921 and gemma_test['tpr_at_fpr_1pct'] > 0.7243:
        print("\u2705 WINNER: EmbeddingGemma 2 outperforms SigLIP-TAP on sealed test.")
        print("   Next step: Prototype MediaPipe UniversalEmbedder Android task in Slot 6.")
    elif gemma_test['auc'] > 0.8919:
        print("\u26a0\ufe0f PARTIAL: EmbeddingGemma 2 beats SigLIP linear probe, but NOT SigLIP-TAP.")
        print("   Recommendation: Do NOT bloat Android APK by +300 MB without beating SigLIP-TAP.")
    else:
        print("\u274c REJECTED: EmbeddingGemma 2 does not beat SigLIP-B/16 linear probe.")
        print("   Retain shipped SigLIP-B/16 probe (89.96 MB INT8) with zero extra bloat.")
else:
    print("Run Step 5 first to produce the evaluation report.")